In [ ]:
!pip install torchinfo

## Importing the relevant libraries

In [ ]:
import torch, warnings, time
warnings.filterwarnings('ignore')
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, mean_squared_log_error
from torchinfo import summary
from torch.nn.utils import clip_grad_norm_

## Loading the Global Equity dataset

In [ ]:
df = pd.read_csv('/kaggle/input/global-equity-dataset/Global Equity.csv',index_col='Unnamed: 0',usecols=['Unnamed: 0','close'])
df.head()

## Data Exploration

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.index.freq = 'W'

In [ ]:
df.index

In [ ]:
df.plot(figsize=(10,5),title='Global Equity');

In [ ]:
df.isna().sum()

In [ ]:
y = df.close.astype('float').values
y

## Dividing data into train and test sets

In [ ]:
train_df, test_df = train_test_split(df,test_size=0.8,shuffle=False,random_state=101)

In [ ]:
train_df.shape, test_df.shape

## Feature Scaling

In [ ]:
scaler = MinMaxScaler(feature_range=(0,1))
train_norm = scaler.fit_transform(train_df)
test_norm = scaler.transform(test_df)

## Creating a custom Time Series Dataset class

In [ ]:
class TimeSeriesDataset(Dataset):

    def __init__(self,sequence_data,sequence_length):
        super().__init__()
        self.sequence = torch.tensor(sequence_data,dtype=torch.float32)
        self.sequence_length = sequence_length
        self.num_samples = len(self.sequence) - self.sequence_length

    def __len__(self):
        return self.num_samples

    def __getitem__(self,idx):
        window = self.sequence[idx:idx+self.sequence_length]
        label = self.sequence[idx+self.sequence_length]
        return (window,label)

In [ ]:
torch.manual_seed(48)
train_data = TimeSeriesDataset(train_norm,sequence_length=10)
test_data = TimeSeriesDataset(test_norm,sequence_length=10)

## Creating Data Loaders for train and test sets

In [ ]:
train_data_loader = DataLoader(dataset=train_data,batch_size=10,shuffle=False,num_workers=5)
test_data_loader = DataLoader(dataset=test_data,batch_size=10,shuffle=False,num_workers=5)

In [ ]:
for seq, targets in train_data_loader:
    break

In [ ]:
seq[0], targets[0]

## Defining the LSTM model architecture

In [ ]:
class LSTM(nn.Module):

    def __init__(self,input_size=1,hidden_size=100,output_size=1):
        super(LSTM,self).__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size=input_size,hidden_size=hidden_size,batch_first=True)
        self.linear = nn.Linear(in_features=hidden_size,out_features=output_size)
        self.hidden = (torch.zeros(1,1,self.hidden_size),torch.zeros(1,1,self.hidden_size))

    def forward(self,sequence):
        batch_size = sequence.size(0)
        self.hidden = self.init_hidden(batch_size)
        lstm_output, self.hidden = self.lstm(sequence,self.hidden)
        pred = self.linear(lstm_output[:,-1,:])
        return pred

    def init_hidden(self,batch_size):
        return (torch.zeros(1,batch_size,self.hidden_size),torch.zeros(1,batch_size,self.hidden_size))

In [ ]:
torch.manual_seed(48)
model = LSTM(input_size=1,hidden_size=150)
model

In [ ]:
total_params = 0

for param in model.parameters():
    print(param.numel())
    total_params += param.numel()

print("Total LSTM model parameters:", total_params)

In [ ]:
summary(model)

## Model Training & Evaluation

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.AdamW(params=model.parameters(),lr=0.001)

In [ ]:
epochs = 100

losses = []

start_time = time.time()

for i in range(1,epochs+1):
    epoch_loss = 0
    
    for seq, y_train in train_data_loader:
        optimizer.zero_grad()
        y_pred = model(seq)
        loss = criterion(y_pred,y_train)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    losses.append(epoch_loss/len(train_data_loader))
    print(f"Epoch {i} -> Loss: {epoch_loss/len(train_data_loader)}")

print(f"Total time elapsed in LSTM model training: {(time.time()-start_time)/60} minutes.")

In [ ]:
plt.plot(range(1,epochs+1),losses)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Loss Graph')
plt.show();

In [ ]:
model.eval() # Set model to evaluation mode
test_loss = 0
all_predictions = []
all_targets = []

with torch.no_grad():

    for seq, y_test in test_data_loader:
        y_pred = model(seq)
        loss = criterion(y_pred,y_test)
        test_loss += loss.item()
        all_predictions.append(y_pred)
        all_targets.append(y_test)

test_loss /= len(test_data_loader)
all_predictions = np.concatenate(all_predictions)
all_targets = np.concatenate(all_targets)
print("Average test loss:", round(test_loss,4))

In [ ]:
all_predictions[:5]

In [ ]:
all_targets[:5]

In [ ]:
true_predictions = scaler.inverse_transform(all_predictions.reshape(-1,1))
true_predictions[:5]

In [ ]:
actual_values = scaler.inverse_transform(all_targets.reshape(-1,1))
actual_values[:5]

In [ ]:
print("MAE:", mean_absolute_error(actual_values,true_predictions))
print("MSE:", mean_squared_error(actual_values,true_predictions))
print("MAPE:", mean_absolute_percentage_error(actual_values,true_predictions))
print("MSLE:", mean_squared_log_error(actual_values,true_predictions))
print("RMSE:", np.sqrt(mean_squared_error(actual_values,true_predictions)))

In [ ]:
test_df.index

In [ ]:
dates = pd.date_range(start='2004-02-06',end='2021-01-29',freq='W')
dates = dates[:len(true_predictions)]
dates

In [ ]:
true_predictions = true_predictions.flatten()
actual_values = actual_values.flatten()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(dates, actual_values, label='Actual Values', color='blue')
plt.plot(dates, true_predictions, label='Predicted Values', color='orange')
plt.xlabel('Date')
plt.ylabel('Values')
plt.title('Time Series Forecast: Predictions vs Actuals')
plt.legend()
plt.grid(True)
plt.show()

## Re-training the model on whole data

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)  # Lower learning rate
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)  # Reduce LR every 20 epochs

In [ ]:
# Custom function to initialize model weights using Xavier initialization
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.LSTM):
        for name, param in m.named_parameters():
            if "weight" in name:
                nn.init.xavier_uniform_(param)
            elif "bias" in name:
                nn.init.zeros_(param)

In [ ]:
model.apply(init_weights)

In [ ]:
epochs = 100

y_norm = scaler.transform(y.reshape(-1,1))
all_data = TimeSeriesDataset(sequence_data=y_norm,sequence_length=10)
all_data_loader = DataLoader(dataset=all_data,batch_size=10,shuffle=False,num_workers=5)

model.train()

losses = []

start_time = time.time()

for i in range(1,epochs+1):

    epoch_loss = 0
    
    for seq, y_train in all_data_loader:
        optimizer.zero_grad()
        y_pred = model(seq)
        loss = criterion(y_pred,y_train)
        loss.backward()
        clip_grad_norm_(parameters=model.parameters(),max_norm=1.0) # Gradient clipping
        optimizer.step()
        epoch_loss += loss.item()

    epoch_loss /= len(all_data_loader)
    scheduler.step() # Adjust learning rate
    losses.append(epoch_loss)
    print(f"Epoch {i} -> Loss: {epoch_loss}")

print(f"Total time elapsed in LSTM model retraining: {(time.time()-start_time)/60} minutes.")

In [ ]:
plt.plot(range(1,epochs+1),losses)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Loss Graph')
plt.show();

## Forecasting into unknown future

In [ ]:
forecast_horizon = 120
sequence_length = 20
predictions = y_norm[-sequence_length:].flatten().tolist()

model.eval()

for i in range(forecast_horizon):
    sequence = torch.FloatTensor(predictions[-sequence_length:]).view(1, sequence_length, 1)

    with torch.no_grad():
        predictions.append(model(sequence).item())

In [ ]:
len(predictions)

In [ ]:
predictions[-sequence_length:][:5]

In [ ]:
true_predictions = scaler.inverse_transform(np.array(predictions).reshape(-1,1))
true_predictions[:5]

In [ ]:
actual_values = scaler.inverse_transform(np.array(y_norm).reshape(-1,1))
actual_values[:5]

In [ ]:
df.index = pd.to_datetime(df.index)
df.index

In [ ]:
future_dates = pd.date_range(start=df.index[-1]+pd.Timedelta(weeks=1),
                             periods=forecast_horizon,
                             freq='W')
future_dates

In [ ]:
len(future_dates), len(true_predictions[sequence_length:])

In [ ]:
fig = plt.figure(figsize=(10,5))
plt.title('Global Equity Forecast')
plt.ylabel('Global Equity')
plt.grid(visible=True)
plt.autoscale(enable=True,axis='x',tight=True)
plt.plot(df.index,actual_values)
plt.plot(future_dates,true_predictions[sequence_length:])
plt.show();

## Saving the model for future use

In [ ]:
torch.save(model,'global_equity_forecaster.pt')

This is a relatively decent forecast considering the simplicity of the LSTM model. Please upvote my notebook if you really liked my work. In case you have any suggestions regarding my work, you can let me know through the comments section below.